# 🔬 M-2LRF vs. Real BitsAndBytes NF4 QLoRA Empirical Benchmark
**Scientific Side-by-Side Comparison on Google Colab GPU (Tesla T4 / A100)**

This notebook executes an **apples-to-apples controlled benchmark** comparing:
1. **Real BitsAndBytes NF4 (4-bit)** + HuggingFace `peft` LoRA (Standard QLoRA)
2. **M-2LRF 2-Bit Dual-Basis Packed** + LoftQ SVD Residual LoRA

**Metrics Evaluated:**
- Base Model Weight Memory (MB) & Compression Ratio
- Peak Training VRAM (MB)
- Step-0 Initial Representation Loss & Step-N Convergence Loss
- WikiText-2 Validation Perplexity (PPL)
- Autoregressive Decoding Latency & Generation Throughput (tokens/s)

In [ ]:
# 1. Install Required Dependencies
!pip install -q transformers bitsandbytes peft accelerate datasets triton matplotlib

In [ ]:
# 2. Check GPU Environment & Hardware Capabilities
import torch
print("=" * 60)
print(f"CUDA Available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")
    print(f"Physical VRAM  : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
print("=" * 60)

In [ ]:
# 3. Clone / Import M-2LRF Production Codebase
import os, sys
if not os.path.exists("m2lrf"):
    !git clone https://github.com/MD-Mushfiqur123/m2lrf.git
    sys.path.insert(0, os.path.abspath("m2lrf"))
else:
    sys.path.insert(0, os.path.abspath("."))

from m2lrf.layer import M2LRF2BitLinear
from m2lrf.trainer_eval import prepare_m2lrf_model
from m2lrf.triton_kernel import HAS_TRITON, m2lrf_triton_matmul
print("✅ M-2LRF Engine successfully imported!")

In [ ]:
# 4. Execute Full Apples-to-Apples Empirical Benchmark (GPT-2 or Qwen2.5-7B)
# Use --model-id gpt2 for quick run, or --model-id Qwen/Qwen2.5-7B-Instruct for 7B run
!python benchmarks/m2lrf_vs_real_qlora_harness.py --model-id gpt2 --rank 16 --steps 50 --batch-size 4

In [ ]:
# 5. Numerical Equivalence & Triton In-SRAM GEMM Speedup Verification
!python benchmarks/verify_triton_gemm.py